# TasteAI: Music Recommendation System
### Audio Feature Engineering, Dynamic Inspection & NearestNeighbors
This notebook demonstrates audio attribute scaling, genre/artist metadata vectorization, and K-Nearest Neighbors recommendation.

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
from ml.preprocessing.music_preprocessing import preprocess_music
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# 1. Load Music Dataset
dataset_path = '../../datasets/music/dataset.csv'
df, scaled_audio, scaler, audio_cols = preprocess_music(dataset_path)
print('Total Tracks:', len(df))
print('Audio Features:', audio_cols)
df.head(3)

In [ ]:
# 2. TF-IDF on Genre + Artist
metadata_text = df['genre'] + ' ' + df['artist']
tfidf = TfidfVectorizer(max_features=500)
tfidf_vecs = tfidf.fit_transform(metadata_text).toarray()

# 3. Combine Scaled Features
combined = np.hstack([scaled_audio * 0.6, tfidf_vecs * 0.4])
nn = NearestNeighbors(n_neighbors=10, metric='cosine')
nn.fit(combined)

In [ ]:
# 4. Recommend Similar Tracks
def recommend_music(song_title, top_k=5):
    matches = df[df['title'].str.lower() == song_title.lower()]
    if matches.empty:
        return f'Song {song_title} not found.'
    idx = matches.index[0]
    distances, indices = nn.kneighbors([combined[idx]], n_neighbors=top_k + 1)
    
    print(f'Top recommendations for "{df.iloc[idx]["title"]}" by {df.iloc[idx]["artist"]}:')
    for rank, (dist, i) in enumerate(zip(distances[0][1:], indices[0][1:]), 1):
        track = df.iloc[i]
        print(f'{rank}. {track["title"]} - {track["artist"]} ({track["genre"]}) | Sim: {1 - dist:.3f}')

recommend_music('Starboy')